---

### 🎓 **Professor**: Apostolos Filippas

### 📘 **Class**: AI Engineering

### 📋 **Topic**: You Can Just Build Things

🚫 **Note**: You are not allowed to share the contents of this notebook with anyone outside this class without written permission by the professor.

---

## Welcome!

In our firstfour lectures, we've covered how
1. We can call LLMs via APIs and get structured responses
2. We can build lexical search with BM25
3. We can build semantic search with embeddings
4. We can combine lexical and semantic search into hybrid search

Today you will put it all together by building a Retrieval Augmented Generation (RAG) system.
- This is a question-answering bot that can answer questions about Fordham University
- You will use real data scraped from the Fordham website.


Your RAG pipeline will look like this:

```
User Question
     ↓
1. RETRIEVE: Find relevant documents (search!)
     ↓
2. AUGMENT: Stuff those documents into a prompt
     ↓
3. GENERATE: Ask an LLM to answer using the context
     ↓
Answer
```


---

# 1. Look at your data

In `data/fordham-website.zip` you'll find **~9,500 Markdown files** scraped from Fordham's website. Each file is one page — admissions info, program descriptions, faculty pages, financial aid, campus life, and more.

Your task: **look at the data**
- The first step in any AI engineering or data science project should always be to familiarize yourself with the data.
- I cannot stress this enough.. without this step, it's hard to build anything useful.

Tips:
- Unzip the archive and look at some of the files. 
- Open a few in a text editor. 
- Get a feel for what you're working with.
- The first line of every file is always the **URL** of the page it was scraped from. The rest is the page content converted to Markdown. Here's an example — `gabelli-school-of-business_veterans.md`:

```markdown
https://www.fordham.edu/gabelli-school-of-business/veterans

# Military Veterans & Active Duty Members of the Military

## Transform Your Knowledge & Skills Into a Business Career for the Future

As a veteran or an active duty member of the United States Armed Services,
you have gained or are currently acquiring the invaluable organizational,
leadership, analytics, and technical knowledge and skills that hiring
managers seek. These transferrable skills provide a major advantage in
emerging, business-related industries where innovation, a global mind-set,
and the ability to lead individuals and teams in the continuously evolving
work environment, are critical for success.

By completing a graduate or undergraduate business degree at the Gabelli
School of Business, you can prepare for a lifelong career in some of
today's fastest-growing fields. ...

### Study at a Top-Ranked, Military-Friendly University

The Gabelli School of Business is part of Fordham University, the only
New York City university to be among those ranked "Best for Vets" by
Military Times. ...

### Learn How the Yellow Ribbon Program Works

The Yellow Ribbon GI Education Enhancement Program, or the Yellow Ribbon
Program, is a part of the Post-9/11 Veterans Educational Assistance Act
of 2008. ...
```

The filenames mirror the URL structure — underscores replace path separators (e.g. `gabelli-school-of-business_veterans.md` came from `/gabelli-school-of-business/veterans`). Some files are short (a few lines), others are quite long.

- Once you've looked around, load the files into Python. Python's built-in `zipfile` module can read zip archives without extracting to disk. Load them into a list of dictionaries or a DataFrame with at least two fields: the filename (or a clean page name) and the content

In [4]:
import os 
from typing import List, Dict

def read_md_files(folder_path: str) -> List[Dict]:
    """
    Read all .md files in a folder.

    PARAMETERS:
    - folder_path: path to directory containing markdown files

    RETURNS:
    - List of dictionaries:
        {
            "doc_id": filename,
            "text": full file content
        }
    """

    documents = []

    # Iterate through all files in folder
    for filename in os.listdir(folder_path):

        # Only process .md files
        if filename.endswith(".md"):

            file_path = os.path.join(folder_path, filename)

            # Open file safely - use errors='replace' to handle any encoding (invalid bytes become �)
            with open(file_path, "r", encoding="utf-8", errors="replace") as f:
                text = f.read()

            documents.append({
                "doc_id": filename,
                "text": text
            })

    return documents

In [5]:
folder_path = "fordham-website-windows"
fordham_all_doc = read_md_files(folder_path)
print(len(fordham_all_doc))

9560


---

# 2. Chunk the Documents

Some of the pages could be too long to embed as a single unit. Down the line, the pages may be too long to stuff into the LLM's prompt during the generation step. As such, most of the RAG systems will break down big documents into into smaller **chunks**.

> 📚 **TERM: Chunking**  
> Splitting documents into smaller, self-contained pieces for embedding and retrieval. The goal is chunks that are small enough to be specific, but large enough to be meaningful.

Your task: **write a function that splits each document into chunks.**

Things to think about:
- What's a reasonable chunk size? (Think about what fits in a prompt vs. what's too vague)
- Should you split on sentences? Paragraphs? A fixed character/word count?
- Should chunks overlap? What happens if an answer spans two chunks?
- How do you keep track of which document each chunk came from? You may need that information down the line.

In [7]:

def chunk_by_paragraphs(
    doc_text: str,
    doc_id: str,
    max_chars: int = 3500,
    overlap_chars: int = 300,
    paragraph_separator: str = "\n\n"
) -> List[Dict]:
    """
    Chunk a single document (web-scraped text) by paragraphs and return a list of dicts.

    WHY paragraph-based:
    - Web-scraped content is often naturally separated into paragraphs.
    - We pack multiple paragraphs into one chunk until we hit max_chars.

    PARAMETERS
    - doc_text: full text of ONE document/page/file
    - doc_id: identifier for this document (e.g., filename or URL hash)
    - max_chars: maximum characters allowed per chunk (approx proxy for token budget)
    - overlap_chars: number of characters to carry over from the end of the previous chunk
                    into the beginning of the next chunk (helps boundary context)
    - paragraph_separator: how paragraphs are separated in your scraped text

    RETURNS
    - List of dictionaries, each dict = one chunk with metadata:
      {
        "doc_id": ...,
        "chunk_id": ...,
        "chunk_index": ...,
        "text": ...
      }
    """

    # 1) Split into paragraphs (remove empty blocks)
    raw_paragraphs = doc_text.split(paragraph_separator)
    paragraphs = [p.strip() for p in raw_paragraphs if p.strip()]

    chunks: List[Dict] = []
    current_chunk = ""
    chunk_index = 0

    def add_chunk(text: str) -> None:
        """Helper to append a finalized chunk with metadata."""
        nonlocal chunk_index #chunk_index is from the outer function
        cleaned = text.strip()
        if not cleaned:
            return
        chunks.append({
            "doc_id": doc_id,
            "chunk_id": f"{doc_id}_chunk_{chunk_index}",
            "chunk_index": chunk_index,
            "text": cleaned
        })
        chunk_index += 1

    # 2) Walk through paragraphs, packing them into chunks up to max_chars
    for para in paragraphs:
        # Re-add spacing so chunks remain readable
        para_with_space = para + "\n\n"

        # 2A) Edge case: a single paragraph is longer than max_chars.
        # We split that paragraph into pieces of size max_chars so we NEVER exceed max_chars.
        if len(para_with_space) > max_chars:
            # First flush whatever we already accumulated as a chunk
            if current_chunk.strip():
                add_chunk(current_chunk)
                current_chunk = ""

            # Split the long paragraph into fixed-size character windows
            start = 0
            while start < len(para_with_space):
                piece = para_with_space[start:start + max_chars]
                add_chunk(piece)
                start += max_chars

            # Continue to next paragraph
            continue

        # 2B) Normal case: try adding paragraph to current chunk
        if len(current_chunk) + len(para_with_space) <= max_chars:
            current_chunk += para_with_space
        else:
            # Current chunk is full -> finalize it
            add_chunk(current_chunk)

            # Start a new chunk, optionally prefixing overlap from the previous chunk
            if overlap_chars > 0:
                overlap_text = current_chunk[-overlap_chars:]
            else:
                overlap_text = ""

            current_chunk = overlap_text + para_with_space

    # 3) Flush any remaining text after loop
    if current_chunk.strip():
        add_chunk(current_chunk)

    return chunks


In [8]:
def chunk_all_documents(documents: list[dict]) -> list[dict]:
    all_chunks = []

    for doc in documents:
        doc_id = doc["doc_id"]
        text = doc["text"]

        # Call your existing single-document function
        chunks = chunk_by_paragraphs(
            doc_text=text,
            doc_id=doc_id
        )

        all_chunks.extend(chunks)

    return all_chunks

In [9]:
all_chunks = chunk_all_documents(fordham_all_doc)

print(f"Total chunks created: {len(all_chunks)}")
print(all_chunks[0])
print(len(all_chunks))

Total chunks created: 19163
{'doc_id': '0001144e6d954f94682637e541ad5d7f.md', 'chunk_id': '0001144e6d954f94682637e541ad5d7f.md_chunk_0', 'chunk_index': 0, 'text': "https://www.fordham.edu/about/living-the-mission/center-on-religion-and-culture/duffy-fellows-program/past-duffy-fellows/2021-2022-duffy-fellows\n\n# 2021-2022 Duffy Fellows\n\n**Afrah Bandagi (FCLC 2023)**\n\n“[Supera las fronteras (Transcend Borders): Spirituality and Migration Activism](https://youtu.be/FHzAAH6iP40?si=wvNcQl9Tlman6nRZ)” (research presentation)\n\n**Major:** Philosophy and Political Science\n\n**Minor:** Peace and Justice Studies\n\nAfrah Bandagi and her research partner, Madeline Hilf, examined how religion and spirituality inform the lives of activists working with migrants at the United States-Mexico border. Afrah first became interested in the global immigration crisis when she was involved with refugee work in Jordan. Bandagi and Hilf's project culminated in an online database consisting of video inte

---

# 3. Embed the Chunks

Now we need to turn each chunk into a vector so we can search over them. You've done this before in Lecture 4.

Your task: **embed all chunks using an embedding model.**

Tips:
- You could use a local model, or API model. What are the tradeoffs?
- This will take a while if you do it serially. You might want to use async/batch.
- Once you've created your embeddings, you may want to save them to disk so you don't have to redo this step every time
- You'll need to embed queries with the **same model** at search time

In [10]:
# Placeholder for your implementation

import os
import time
import numpy as np
from openai import OpenAI

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

def embed_texts_openai(
    texts: list[str],
    model: str = "text-embedding-3-small",
    batch_size: int = 128,
    max_retries: int = 6,
    sleep_base: float = 1.0,
) -> np.ndarray:
    """
    Returns: embeddings array shape (N, D) float32
    """
    all_vecs: list[list[float]] = []

    for start in range(0, len(texts), batch_size):
        batch = texts[start : start + batch_size]

        # basic guard: remove empty strings (API rejects empty input)
        batch = [t for t in batch if t and t.strip()]
        if not batch:
            continue

        # retry with exponential backoff
        for attempt in range(max_retries):
            try:
                resp = client.embeddings.create(model=model, input=batch)
                # resp.data is aligned with input order
                all_vecs.extend([item.embedding for item in resp.data])
                break
            except Exception as e:
                # backoff
                if attempt == max_retries - 1:
                    raise
                sleep_s = sleep_base * (2 ** attempt)
                time.sleep(sleep_s)

    return np.array(all_vecs, dtype=np.float32)

def embed_chunks_openai(
    chunks: list[dict],
    text_key: str = "text",
    model: str = "text-embedding-3-small",
    batch_size: int = 128,
):
    texts = [c[text_key] for c in chunks]
    embeddings = embed_texts_openai(texts, model=model, batch_size=batch_size)
    return embeddings

# Example:
embeddings = embed_chunks_openai(all_chunks, model="text-embedding-3-small", batch_size=128)

# Save
np.save("fordham_embeddings.npy", embeddings)
np.save("fordham_chunks_meta.npy", np.array(
    [{"chunk_id": c["chunk_id"], "doc_id": c["doc_id"]} for c in all_chunks],
    dtype=object
))

In [11]:
print("Number of chunks:", len(all_chunks))
print("Number of embeddings:", len(embeddings))

Number of chunks: 19163
Number of embeddings: 19163


---

# 4. Retrieve

Now build the **R** in RAG. Given a user's question, find the most relevant chunks.

Your task: **write a retrieval function that takes a question and returns the most relevant chunks.**

Tips:
- You can use lexical or semantic search or both!
- How many chunks should you retrieve? Too few and you might miss the answer; too many and you'll overwhelm the LLM (and pay more tokens)
- Try a few test questions and eyeball whether the retrieved chunks are relevant
- Try a few questions and see what comes back. For example:
  - "What programs does the Gabelli School of Business offer?"
  - "How do I apply for financial aid?"
  - "Where is Fordham's campus?"

### Semantic 

In [12]:
# Your implementation here

def batch_cosine_similarity(query_vec: np.ndarray, matrix: np.ndarray) -> np.ndarray:
    """
    query_vec: (D,)
    matrix: (N, D)
    returns: (N,)
    """
    q = query_vec.astype(np.float32)
    M = matrix.astype(np.float32)

    q_norm = np.linalg.norm(q) + 1e-12
    M_norm = np.linalg.norm(M, axis=1) + 1e-12

    return (M @ q) / (M_norm * q_norm)

In [13]:
import pandas as pd
from helpers import (
    # BM25 
    build_index, score_bm25,_get_stemmer, snowball_tokenize,
    # Embeddings
    get_local_model, batch_embed_local,
    # Similarity
    batch_cosine_similarity,
    # Utility
    normalize_scores
)

def semantic_search_api(query: str, chunk_embeddings: np.ndarray, chunks_df: pd.DataFrame, embed_query_fn, k: int = 10):
    """
    Search chunks using semantic similarity
    """
    # 1) Embed the query
    query_emb = embed_query_fn(query)  # (D,) numpy array

    # 2) Similarity to all chunks
    similarities = batch_cosine_similarity(query_emb, chunk_embeddings)  # (N,)

    # 3) Top-k indices
    top_k_idx = np.argsort(-similarities)[:k]

    # 4) Results DF
    results = chunks_df.iloc[top_k_idx].copy()
    results["similarity"] = similarities[top_k_idx]
    results["rank"] = range(1, k + 1)

    # Optional: reorder columns nicely
    cols = [c for c in ["rank", "doc_id", "chunk_id", "similarity", "text"] if c in results.columns]
    return results[cols]

In [14]:


client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

def embed_query_openai(query: str, model: str = "text-embedding-3-small") -> np.ndarray:
    response = client.embeddings.create(
        model=model,
        input=[query]
    )
    return np.array(response.data[0].embedding, dtype=np.float32)

In [15]:
chunks_df=pd.DataFrame(all_chunks)
results = semantic_search_api(
    query="How do I apply for financial aid?",
    chunk_embeddings=embeddings,
    chunks_df=chunks_df,
    embed_query_fn=embed_query_openai,
    k=10
)

results[["rank", "doc_id", "chunk_id", "similarity"]]

,rank,doc_id,chunk_id,similarity
10103,1,8034e09c4cc5530aebcb4e2bcf39ef94.md,8034e09c4cc5530aebcb4e2bcf39ef94.md_chunk_2,0.624265
1584,2,1517a7222213dd6fa224e2760a9c06f5.md,1517a7222213dd6fa224e2760a9c06f5.md_chunk_1,0.584795
17049,3,e4214b9178885f25eb8f993d914a390e.md,e4214b9178885f25eb8f993d914a390e.md_chunk_0,0.583533
1583,4,1517a7222213dd6fa224e2760a9c06f5.md,1517a7222213dd6fa224e2760a9c06f5.md_chunk_0,0.575792
6156,5,487f8f1ebf1c62c7c2c1cfb423c52587.md,487f8f1ebf1c62c7c2c1cfb423c52587.md_chunk_0,0.573225
12395,6,9f0ca74dca65ab25525efb18032cf514.md,9f0ca74dca65ab25525efb18032cf514.md_chunk_1,0.569678
10537,7,85b055c14303a8fc151396df9088d62e.md,85b055c14303a8fc151396df9088d62e.md_chunk_1,0.568886
11137,8,8eaac87cb189927dc2febc989ce5d426.md,8eaac87cb189927dc2febc989ce5d426.md_chunk_1,0.568886
18144,9,f294c5ae01ccc3db6ff6d80eae16b0ab.md,f294c5ae01ccc3db6ff6d80eae16b0ab.md_chunk_1,0.568833
18153,10,f2bbc1f96d866f2ec8f18d5ba58a2300.md,f2bbc1f96d866f2ec8f18d5ba58a2300.md_chunk_2,0.561045


### Lexical

In [16]:
from helpers import (
    # BM25 
    build_index, _get_stemmer, snowball_tokenize,
    # Utility
    normalize_scores
)


In [18]:
lexdoc = [chunk['text'] for chunk in all_chunks]

bm25_index, lexdoc_length = build_index(lexdoc)

In [19]:
def lexical_search(query, k=5):
    scores = score_bm25(
        query=query,
        index=bm25_index,
        num_docs=len(lexdoc),
        doc_lengths=lexdoc_length 
    )
    top_idx = np.argsort(-scores)[:k]
    return chunks_df.iloc[top_idx].assign(bm25_score=scores[top_idx])

In [54]:
def hybrid_search(query, chunks_df, chunk_embeddings, bm25_index, lexdoc, lexdoc_length, alpha=0.5, k=5):
    # semantic
    q_emb = embed_texts_openai([query], model = "text-embedding-3-small").reshape(-1)
    sem_scores = batch_cosine_similarity(q_emb, chunk_embeddings)

    # lexical
    lex_scores = score_bm25(
        query, bm25_index, len(lexdoc), lexdoc_length
    )

    # normalize
    sem_norm = normalize_scores(sem_scores)
    lex_norm = normalize_scores(lex_scores)

    final_scores = alpha * sem_norm + (1 - alpha) * lex_norm

    top_idx = np.argsort(-final_scores)[:k]
    return chunks_df.iloc[top_idx].assign(
        hybrid_score=final_scores[top_idx],
        sem_score=sem_norm[top_idx],
        lex_score=lex_norm[top_idx],
    )

In [55]:
query_hybrid = "school libraries"
hybrid_results = hybrid_search(query_hybrid, chunks_df, embeddings, bm25_index, lexdoc, lexdoc_length,alpha=0.5)

print(f"Hybrid search for '{query_hybrid}':")
print(hybrid_results)

Hybrid search for 'school libraries':
                                    doc_id  \
16756  e00252c7e1bdb2ce3f98e77573803a38.md   
11106  8de46c3644de0381322d372002638a0a.md   
6747   50c05c30a938be9a1a46c754670c9bda.md   
11352  925907f22b4c5c7a357c0072cc914381.md   
295    03ab08ed7abd6e8e7de175b6aec89e68.md   

                                          chunk_id  chunk_index  \
16756  e00252c7e1bdb2ce3f98e77573803a38.md_chunk_0            0   
11106  8de46c3644de0381322d372002638a0a.md_chunk_0            0   
6747   50c05c30a938be9a1a46c754670c9bda.md_chunk_0            0   
11352  925907f22b4c5c7a357c0072cc914381.md_chunk_0            0   
295    03ab08ed7abd6e8e7de175b6aec89e68.md_chunk_0            0   

                                                    text  hybrid_score  \
16756  https://www.fordham.edu/resources/libraries/re...      0.985295   
11106  https://www.fordham.edu/resources/libraries/ab...      0.956732   
6747   https://www.fordham.edu/resources/libraries\n\...    

---

# 5. Generate

Now build the **G** in RAG. Take the retrieved chunks and pass them to an LLM along with the user's question.

Your task: **write a function that takes a question and the retrieved chunks, builds a prompt, and calls an LLM to generate an answer.**

Tips:
- How should you structure the prompt? The LLM needs to know: (1) what is the context of the application, (2) what is the question, (3) what it should include in its answer
- What should the LLM do if the context doesn't contain the answer?
- Start with a cheap model; try a better one when you've figured out the pipeline

In [ ]:
# Placeholder for your implementation
""" def get_retrieved_text(results_df, chunks_df, top_k=10):
    top = results_df.sort_values("rank").head(top_k).copy()

    # Join để lấy text
    merged = top.merge(
        chunks_df[["chunk_id", "text", "doc_id"]],
        on=["chunk_id", "doc_id"],
        how="left"
    )

    # Convert to list-of-dict
    retrieved_chunks = []
    for _, r in merged.iterrows():
        retrieved_chunks.append({
            "chunk_id": r["chunk_id"],
            "doc_id": r["doc_id"],
            "text": r["text"],
            "similarity": float(r["similarity"]),
            "rank": int(r["rank"]),
        })
    return retrieved_chunks """

In [ ]:
def get_retrieved_chunks(results_df, top_k=10):
    top = results_df.sort_values("rank").head(top_k).copy()

    retrieved_chunks = []
    for _, r in top.iterrows():
        retrieved_chunks.append({
            "chunk_id": r["chunk_id"],
            "doc_id": r["doc_id"],
            "text": r["text"],              
            "similarity": float(r["similarity"]),
            "rank": int(r["rank"]),
        })
    return retrieved_chunks

In [22]:

def generate_answer(question, retrieved_chunks):
    context = "\n\n".join([c["text"] for c in retrieved_chunks])

    prompt = f"""
Answer the question using ONLY the context below.

Question:
{question}

Context:
{context}
"""

    response = client.responses.create(
        model="gpt-4.1-mini",
        input=prompt
    )

    return response.output_text

In [50]:
results = semantic_search_api(
    query="Graduate Programs at Fordham",
    chunk_embeddings=embeddings,
    chunks_df=chunks_df,
    embed_query_fn=embed_query_openai, k=10)
retrieved_chunks = get_retrieved_chunks(results, top_k=10)

answer = generate_answer(question="Graduate Programs at Fordham", retrieved_chunks=retrieved_chunks)
print(answer)

Graduate Programs at Fordham University include:

- More than 130 graduate degree programs with a strong focus on research, mentorship by respected faculty, and curricula reflecting new industries and global challenges.
- Over 25 accelerated degree options available.
- Programs housed across 8 schools/colleges on 3 campuses (Lincoln Center, Rose Hill, Westchester), including:
  - Fordham College at Lincoln Center
  - Gabelli School of Business
  - Graduate School of Education
  - Graduate School of Social Service
  - School of Law
  - Graduate School of Arts and Sciences
  - School of Professional and Continuing Studies

Specific options include:
- Dual-degree and early admission programs for promising undergraduates at Fordham College at Lincoln Center, including 3-3 program with the School of Law, accelerated master's in English, MSW in one year, and accelerated MS and teaching certification.
- Graduate School of Education programs such as Doctor of Education Administration and Super

### Generate answer for hybrid search

In [40]:
def get_retrieved_chunks_hybrid(hybrid_df, top_k=5, score_col="hybrid_score", text_col="text"):
    df = hybrid_df.copy()

    # sort by hybrid score (cao -> tốt)
    top = df.sort_values(score_col, ascending=False).head(top_k).copy()

    retrieved_hybrid = []
    for rank_idx, (_, r) in enumerate(top.iterrows(), start=1):
        retrieved_hybrid.append({
            "chunk_id": r.get("chunk_id"),
            "doc_id": r.get("doc_id"),
            "text": r.get(text_col),
            "similarity": float(r.get(score_col)),  
            "rank": rank_idx,
            # optional for debug
            #"hybrid_score": float(r.get(score_col)),
            #"sem_score": float(r.get("sem_score")) if "sem_score" in r else None,
            #"lex_score": float(r.get("lex_score")) if "lex_score" in r else None,
        })
    return retrieved_hybrid

In [41]:
retrieved_hybrid = get_retrieved_chunks_hybrid(hybrid_results, top_k=5)
answer_hybrid = generate_answer(query_hybrid, retrieved_hybrid)
print(answer_hybrid)

School libraries at Fordham University are integral parts of the Fordham University Libraries system, which comprises three major locations:

1. **William D. Walsh Family Library at Rose Hill (Bronx campus)**  
   - Contains over 1,000,000 volumes and 380,000 government documents.  
   - Houses all Rose Hill library services including Science Library, Audiovisuals, Electronic Services, Government Documents, Archives, Special Collections, Microforms, and Fordham Dissertations.

2. **Gerald M. Quinn Library at Lincoln Center (Manhattan campus)**  
   - Contains about 500,000 volumes.  
   - Serves Fordham College at Lincoln Center and has strong collections in business, education, and social service to support the graduate schools there.

3. **Fordham Westchester Library (Harrison campus)**  
   - Houses over 30,000 volumes.  
   - Serves the graduate schools of Business, Education, Religious Education, and Social Services.  
   - Provides access to all Fordham electronic books and datab

In [26]:
print(chunks_df.columns.tolist())
print(results.columns.tolist())

['doc_id', 'chunk_id', 'chunk_index', 'text']
['rank', 'doc_id', 'chunk_id', 'similarity', 'text']


---

# 6. Wire everything together

Combine the previous steps into a simple function that takes in a question and returns an answer.

Your task: **write a `rag(question)` function that retrieves relevant chunks and generates an answer.**

In [49]:
# Placeholder for your implementation

def rag(question):
    rag_result = semantic_search_api(
    question,
    chunk_embeddings=embeddings,
    chunks_df=chunks_df,
    embed_query_fn=embed_query_openai, k=5)
    retrieved_ans = get_retrieved_chunks(rag_result, top_k =5)
    rag_answer = generate_answer(question,retrieved_ans)
    return print(rag_answer)

rag("What are the graduate programs in Fordham?")

Fordham University offers more than 130 graduate degree programs across 8 schools/colleges on 3 campuses. The graduate schools and programs include:

- Graduate School of Arts and Sciences
- Graduate School of Education
- Graduate School of Social Service
- Gabelli School of Business
- School of Law
- School of Professional and Continuing Studies

Key features of Fordham's graduate programs are a strong focus on research, mentorship by respected faculty, and curricula that reflect current industries and global challenges.

Special graduate program options for undergraduates at Fordham College at Lincoln Center include:
- Accelerated degree programs leading to MSW, JD, MA, MS, and MST teaching degrees.
- The 3-3 program with Fordham Law School to save a year of study.
- Accelerated master’s program in English.
- One-year MSW degree at the Graduate School of Social Service.
- MS degree and teaching certification at the Graduate School of Education.
- Early admission to select MA and MS d

(query_hybrid, chunks_df, embeddings, bm25_index, lexdoc, lexdoc_length,alpha=0.5)

In [64]:
def rag_hybrid(question):
    rag_result_hybrid = hybrid_search(
    question,
    chunks_df =chunks_df,
    chunk_embeddings=embeddings,
    bm25_index = bm25_index,
    lexdoc = lexdoc,
    lexdoc_length = lexdoc_length,
    alpha =0.5 , k=5)
    retrieved_hybrid = get_retrieved_chunks_hybrid(rag_result_hybrid, top_k =5)
    ans_hybrid = generate_answer(question,retrieved_hybrid)
    return print(ans_hybrid)

rag("What are the graduate programs in Fordham?")

Fordham University offers graduate programs through multiple graduate schools and colleges, with a strong focus on research, mentorship, and curricula addressing current global challenges. The graduate schools and programs include:

- Graduate School of Arts and Sciences
- Graduate School of Education
- Graduate School of Social Service
- Gabelli School of Business
- School of Law
- School of Professional and Continuing Studies

There are over 130 graduate degree programs available. Fordham also offers 25+ accelerated degree options, including dual and early admission programs for qualified undergraduates.

Graduate study at Fordham takes place across three campuses:
- Rose Hill (Bronx)
- Lincoln Center (Manhattan)
- Other sites like Fordham London, Calder Center, and Westchester campus

Some highlighted graduate programs and opportunities include:
- Real Estate and Applied Health Informatics programs
- MSW (Master of Social Work) degree that can be earned in one year at the Graduate S

---

# 7. Evaluate, experiment and improve

Your RAG system works — but there's always room to make it better. 

Your task: **evaluate, experiment, and improve your system**

Tips:
- How do you know that your system is working or that your changes are improving it?
- Try different questions — where does it do well? Where does it struggle?
- Adjust the number of retrieved chunks — what happens with more or fewer?
- Try different chunking strategies — bigger chunks? Smaller? Overlap?
- Try a different embedding model — does it change retrieval quality?
- Improve the prompt — can you get better, more concise answers?
- Add source attribution — can the system tell the user which pages the answer came from?

### Try different questions

In [46]:
# Placeholder for your implementation
rag("Fordham alumni career")

Fordham alumni pursue careers in a variety of industries and benefit from a strong support system and network after graduation:

- Fordham has a network of over 175,000 alumni worldwide who are willing to assist fellow graduates.
- The university's Career Center offers lifelong support including one-on-one appointments, workshops, events, and access to top employers.
- Nearly 90% of Fordham students have at least one internship before graduating, which helps them gain practical experience.
- The Class of 2024 graduates found work in nearly every sector, with leading industries including:
  - Investment Banking (104 graduates)
  - Financial Services (89)
  - Advertising, PR & Marketing (85)
  - Healthcare (62)
  - Accounting (61)
  - Legal & Law Enforcement (56)
  - Non-Profit (54)
  - Media, entertainment, fashion, tech among others.
- Top hiring employers for 2024 graduates include:
  - Fordham University (19)
  - PricewaterhouseCoopers (19)
  - JPMorgan Chase & Co. (17)
  - EY (16)
 

In [65]:
rag_hybrid("Fordham Alumni career")

Fordham alumni have access to a wide range of career resources and support across different schools and programs:

### Gabelli School of Business Alumni Career Resources
- Access Fordham’s networking career platform, RamConnect, for connections and job listings.
- Join the Gabelli School Alumni LinkedIn group.
- Schedule appointments with alumni career advisors (for graduate alumni).
- Discuss career plans with an alumni career coach.
- Request official transcripts.
- Opportunities to audit Gabelli School classes for skill development.
- Participate in mentoring programs like the Fordham Mentoring Program and Rams Helping Rams.
- Attend career development events and workshops organized by the Career Center.
- Network through affinity and regional alumni chapters.
- Access on-campus resources with a digital Ram Pass.

### Fordham Law School Alumni Career Support
- Lifetime access to the Career Planning Center (CPC) with individual counseling sessions.
- Career development programs such 

In [47]:
rag("Business Analytics graduate application deadline?")

The application deadline for the M.S. in Business Analytics program at Fordham University's Gabelli School of Business for Summer 2026 admission is **March 6, 2026**.


In [53]:
rag("MSBA admission deadline for international students")

The MSBA (M.S. in Business Analytics) admission deadline for international students who are outside the U.S. and require a student visa is the Round 3 deadline:

- **Deadline:** March 20, 2026  
- This is the final deadline for all international students outside of the U.S. requiring a student visa.

Applications after this date for international students needing a visa will not be accepted.  

Reference:  
- Fall 2026 Round 3 Final Application Deadline: March 20, 2026  
- Note: "Round 3: Final deadline for all international students outside of the U.S. requiring a student visa"


In [66]:
rag_hybrid("MSBA admission deadline")

The MSBA (Master’s in Business Analytics) admission deadlines for Fall 2026 are as follows:

- Round 1: Final Application Deadline - October 10, 2025  
- Round 2: Final Application Deadline - January 16, 2026  
- Round 3: Final Application Deadline - March 20, 2026  
- Round 4: Rolling Admissions (no final deadline given, for domestic students or international students already in the U.S.)  

Additional notes:
- Round 1 applications have the application fee waived.  
- Round 2 applications get priority consideration for scholarships.  
- Round 3 is the final deadline for all international students outside of the U.S. requiring a student visa.  

This information is from the "Graduate Admissions Deadlines" section relevant to M.S. Programs (on-campus) at Fordham University Gabelli School of Business.


There is some difference between the answers.
1. The difference between answer using different search method - Semantic-only and hybrid search. For semantic-only, the answer is more concise and clear, while the hybrid method answer is longer and it has a lot of "alumni career" keyword that matches with the questions. Due to the question contain only noun phrases but does not show the detailed intention of user, the semantic gives answer about the historical data of alumni career - which shows prospective opportunity for incoming students. However, the hybrid one tend to think the user as alumni and is searching for support.

2. When using the same Semantic-only method, asking the same question of the application deadline for MSBA program, it gives two different answer. For general question, it gives general answer, when the question specifies the applicants being "international students", it gives more detailed information. The same question when given to hybrid search, seem to also have better answer than the semantic-only as it tries to match keywords and indicate more information for different background applicants than just one single deadline. 

=> It seems that the more detailed the question is, the more accurate answer will be given. 


### Adjustment on number of chunks retrieved

In [69]:
query_hybrid = "Student Engagement activity in Gabelli School"
hybrid_results = hybrid_search(query_hybrid, chunks_df, embeddings, bm25_index, lexdoc, lexdoc_length,alpha=0.5, k=5)

print(f"Hybrid search for '{query_hybrid}':")

retrieved_hybrid = get_retrieved_chunks_hybrid(hybrid_results, top_k=5)
answer_hybrid = generate_answer(query_hybrid, retrieved_hybrid)
print(answer_hybrid)

Hybrid search for 'Student Engagement activity in Gabelli School':
Student Engagement activities in the Gabelli School of Business for graduate students focus on enhancing education through leadership, diversity, equity and inclusion, and community service skills. These activities provide valuable experience and complement professional development, serving as important résumé additions.

Key engagement opportunities include:

- **Leadership Development:** Access to leadership resources addressing ethics, sustainability, equitable outcomes, and corporate responsibility to help build critical leadership skills.

- **Student Life:** A variety of activities, organizations, and events that allow students to bond with classmates, explore interests, and build professional networks. Examples of student groups include the Black and Latinx M.B.A. Association, Business Veterans, Women in Business, and Gabelli Pride.

- **Corporate Engagement:** Opportunities such as company visits, networking eve

In [ ]:
query_hybrid = "Student Engagement activity in Gabelli School"
hybrid_results = hybrid_search(query_hybrid, chunks_df, embeddings, bm25_index, lexdoc, lexdoc_length,alpha=0.5, k=10) #chunks retrieved

print(f"Hybrid search for '{query_hybrid}':")

retrieved_hybrid = get_retrieved_chunks_hybrid(hybrid_results, top_k=10) #chunks fed to LLm
answer_hybrid = generate_answer(query_hybrid, retrieved_hybrid)
print(answer_hybrid)

Hybrid search for 'Student Engagement activity in Gabelli School':
Student Engagement activities at the Gabelli School of Business include a comprehensive range of programs designed to enhance graduate students' education and professional development. These activities focus on leadership development, diversity, equity, and inclusion, community service, and building critical personal and professional connections.

Key engagement opportunities for graduate students are:

1. **Leadership Development:** Access to leadership resources that help students understand ethics, sustainability, equitable outcomes, and corporate responsibility.

2. **Student Life Activities:** Orientation events, student organizations (such as the Black and Latinx M.B.A. Association, Business Veterans, Women in Business, and Gabelli Pride), social and networking events, company site visits, and speaker series.

3. **Graduate Student Advisory Council (SAC):** The principal graduate student organization that advocate

In [77]:
query_hybrid = "Student Engagement activity in Gabelli School"
hybrid_results = hybrid_search(query_hybrid, chunks_df, embeddings, bm25_index, lexdoc, lexdoc_length,alpha=0.5, k=20)

print(f"Hybrid search for '{query_hybrid}':")

retrieved_hybrid = get_retrieved_chunks_hybrid(hybrid_results, top_k=20)
answer_hybrid = generate_answer(query_hybrid, retrieved_hybrid)
print(answer_hybrid)

Hybrid search for 'Student Engagement activity in Gabelli School':
Student Engagement activities at the Gabelli School of Business include a rich variety of programs designed to complement education and professional development for graduate students. Key components are:

1. **Leadership Development:** Access to leadership resources addressing ethics, sustainability, equitable outcomes, and corporate responsibility, helping students build valuable leadership qualities.

2. **Student Life Activities:** Orientation, numerous activities, student organizations, and events to foster community, build connections, explore interests, and create networks critical for career success.

3. **Student Clubs:** Over 25 graduate student clubs covering professional, cultural, and affinity interests, such as Black and Latinx M.B.A. Association, Women in Business, Gabelli Pride, Business Analytics Society, and more. These clubs hold guest lectures, networking events, company site visits, and conferences.


Testing on k=5 and k=10 and k=20, the later ones gives more information as it retrieves more chunks and feeds more to the LLM to generate answer. There is clear gap between k-5 and k=10, but the answer between k=10 and k=20 do not differ much.

## Evaluate

Using Synthetic Data because there is not relevance file/ground truth to based on

In [90]:
import litellm
import asyncio
import random
import textwrap
from pydantic import BaseModel, Field


# 'chain_of_thought' makes the LLM reason before generating the question
class SyntheticQuestion(BaseModel):
    chain_of_thought: str = Field(description="Step-by-step reasoning about what makes a good question for this document")
    question: str = Field(description="A natural, specific question that can be answered using the document")
    answer: str = Field(description="The answer to the question")


# random "constraints" 
constraints = [
    "The question should be answerable in a short phrase",
    "The question should require synthesizing multiple facts from the document",
    "Frame the question as something a student might look for on Fordham's website",
]


async def generate_question(doc_id: str,chunk_id: str, text: str) -> dict:
    """Generate a synthetic question for a single document using an LLM."""
    constraint = random.choice(constraints)
    response = await litellm.acompletion(
        model="openai/gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": textwrap.dedent(f"""
                
                I will give you a document retrieved from Forham University's website. Please Generate a question that can be answered using the following document.
                
                Doc: {doc_id}
                Text: {text}
                
                Rules:
                - Your question should be natural and specific and concise
                - Your question should think of questions students or parents may want to look for about Fordham
                - Your question must be answerable using the document that I gave you
                - {constraint}
                - Do not reference \"the document\" or \"the study\" in your question
                """
                ),
            }
        ],
        response_format=SyntheticQuestion,
    )
    
    # Parse the JSON response into our Pydantic model
    result = SyntheticQuestion.model_validate_json(response.choices[0].message.content)
    return {"doc_id": doc_id, "question": result.question, "answer": result.answer}


# Sample 80 documents to generate questions for
sample_docs = chunks_df.sample(n=50, random_state=42)

# Generate all questions concurrently using asyncio.gather
tasks = [generate_question(row["doc_id"],row['chunk_id'], row["text"]) for _, row in sample_docs.iterrows()]
synthetic_results = await asyncio.gather(*tasks)

synthetic_df = pd.DataFrame(synthetic_results)
print(f"Generated {len(synthetic_df)} synthetic questions\n")

# Show some examples with their source documents
for _, row in synthetic_df.head(10).iterrows():
    doc = chunks_df[chunks_df["doc_id"] == row["doc_id"]].iloc[0]
    print(f"Q: {row['question']}")
    print(f"   Source: {doc['doc_id'][:80]}")
    print(f"   Answer: {row['answer']}")
    print()

Generated 50 synthetic questions

Q: What are the admission requirements for the Gabelli School of Business at Rose Hill?
   Source: ca798be26ee789a09746def1a2d23642.md
   Answer: To be eligible to apply to the Gabelli School of Business at Rose Hill, an applicant must have completed high school Precalculus, Calculus, AP Calculus (AB or BC), IB Math, or college-level Algebra, Pre-calculus, Calculus, or Finite Math.

Q: What services does Fordham University offer to support student life?
   Source: 946aca22f850c4021fc70641112bf15a.md
   Answer: Fordham University offers services such as campus culture initiatives, dining and hospitality, safety and wellness services, multicultural affairs, public safety, commuter and disability services, career resources, and opportunities for student involvement.

Q: What courses does Dr. Joshua L. Brown teach that align with his research on school-based interventions in urban education?
   Source: 043bd6c3a3c4f47141dbcccc63d5c5c0.md
   Answer: Dr. Jos

In [ ]:
# Evaluate retrieval on synthetic questions
# Key difference from qrels: each synthetic question has exactly 1 relevant doc (its source)
syn_rows = []

for _, row in synthetic_df.iterrows():
    source_id = row["doc_id"]
    question = row["question"]

    # Search for the synthetic question
    retrieved_synthetic = table.search(question, query_type="vector").limit(max_k).to_list()
    retrieved_ids = [r["doc_id"] for r in retrieved_synthetic]

    for k in k_values:
        ids_at_k = retrieved_ids[:k]

        # Binary relevance: did we find the source document in top-k?
        found = source_id in ids_at_k
        precision = (1.0 if found else 0.0) / k  # at most 1 relevant doc
        recall = 1.0 if found else 0.0  # found it or didn't

        syn_rows.append({"metric": "precision", "k": k, "search_type": "vector", "score": precision, "question": question})
        syn_rows.append({"metric": "recall", "k": k, "search_type": "vector", "score": recall, "question": question})

syn_eval_df = pd.DataFrame(syn_rows)

# Compare with qrels results above — synthetic questions are typically "easier" for retrieval
# because the LLM generates questions using the document's own language
print("Retrieval evaluation (synthetic questions, vector search):\n")
print(syn_eval_df.groupby(["search_type", "metric", "k"])["score"].mean().round(4).to_string())

In [91]:
def evaluate_synthetic(synthetic_df, retrieve_ids_fn, k_values=(1,3,5,10)):
    rows = []
    for _, row in synthetic_df.iterrows():
        source_id = row["doc_id"]
        q = row["question"]

        max_k = max(k_values)
        retrieved_ids = retrieve_ids_fn(q, top_k=max_k)

        for k in k_values:
            ids_at_k = retrieved_ids[:k]
            found = source_id in ids_at_k
            precision = (1.0 if found else 0.0) / k
            recall = 1.0 if found else 0.0

            rows.append({"metric":"precision","k":k,"score":precision,"question":q})
            rows.append({"metric":"recall","k":k,"score":recall,"question":q})

    df = pd.DataFrame(rows)
    return df

In [101]:
def retrieve_ids_hybrid(q, top_k):
    results_df = hybrid_search(
        q,
        chunks_df=chunks_df,
        chunk_embeddings=embeddings,
        bm25_index=bm25_index,
        lexdoc=lexdoc,
        lexdoc_length=lexdoc_length,
        alpha=0.5,
        k=top_k,          # nếu hybrid_search có param này
    )

    if hasattr(results_df, "columns"):
        if "doc_id" in results_df.columns:
            return results_df["doc_id"].head(top_k).tolist()
        if "id" in results_df.columns:
            return results_df["id"].head(top_k).tolist()
    return [r.get("doc_id", r.get("id")) for r in results_df][:top_k]

In [102]:
syn_eval_df = evaluate_synthetic(synthetic_df, retrieve_ids_hybrid, k_values=(1,3,5,10))
print(syn_eval_df.groupby(["metric","k"])["score"].mean().round(4))

metric     k 
precision  1     0.580
           3     0.260
           5     0.168
           10    0.084
recall     1     0.580
           3     0.780
           5     0.840
           10    0.840
Name: score, dtype: float64


In this test, LLM generate question based on one document, therefore the Precision at K=1 gains the highest of all, the more documents there are, the less relevant it is. 
For recall, 58% of questions retrieved correct information at the first position. 78% has their related document at top 3. The number then does not change much because the most relevant document should always be in the top 3. 

---

# 8. (Optional) Make it an app

So far your RAG system lives inside a notebook. That's great for development — but nobody is going to use your Jupyter notebook to ask questions about Fordham. Let's turn it into a real web app.

> 📚 **TERM: Streamlit**  
> A Python library that turns plain Python scripts into interactive web apps. You write Python — no HTML, CSS, or JavaScript — and Streamlit renders it as a web page with inputs, buttons, and formatted output. It's the fastest way to go from "I have a function" to "I have a web app."

Your task: **create a Streamlit app that lets a user type a question about Fordham and get an answer from your RAG system.**

To get started:
- Install it: `uv pip install streamlit` 
- A Streamlit app is just a `.py` file (not a notebook). Create something like `fordham_rag_app.py`
- Run it: `streamlit run scripts/fordham_rag_app.py` — this opens a browser tab with your app

Tips:
- Check out the [Streamlit docs](https://docs.streamlit.io/) — the "Get started" tutorial is very short
- Your best bet is to vibecode your way to this. You'll be surprised how fast you can get it up and running

---

# Summary

## What You Built

| Step | What You Did | What It Does |
|------|-------------|-------------|
| **Load** | Read 9,500+ Fordham web pages | Get raw content |
| **Chunk** | Split pages into smaller pieces | Make content searchable and promptable |
| **Embed** | Turn chunks into vectors | Enable semantic search |
| **Retrieve** | Find relevant chunks for a question | The **R** in RAG |
| **Generate** | Ask an LLM to answer using the chunks | The **G** in RAG |
| **RAG** | Wire it all together | Question in, answer out |

## The Big Picture

RAG is one of the most common patterns in AI engineering today. What you built here is the same core architecture behind tools like ChatGPT with search, Perplexity, enterprise Q&A bots, and more. The details get more sophisticated (vector databases, reranking, query rewriting, evaluation) but the pattern is the same:

**Find relevant stuff → give it to an LLM → get an answer.**

You can just build things.